In [1]:
# ============================================================
# CELL 1 — Imports & Config
# ============================================================
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_USE_FUSED_CROSS_ENTROPY"] = "0"

import unsloth
from unsloth import FastLanguageModel
import torch, gc
import matplotlib.pyplot as plt
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

gc.collect()
torch.cuda.empty_cache()

# Config
MODEL_NAME   = "unsloth/Qwen2.5-3B-Instruct"
DATASET_PATH = "csv/train_clean.csv"
OUTPUT_DIR   = "./qwen-svg-lora-v2"
MAX_SEQ_LEN  = 1024
BATCH_SIZE   = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"VRAM in use: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Old model preserved at: ./qwen-svg-lora")
print(f"New model will save to: {OUTPUT_DIR}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
VRAM in use: 0.00 GB
GPU: NVIDIA GeForce RTX 3090
Old model preserved at: ./qwen-svg-lora
New model will save to: ./qwen-svg-lora-v2


In [3]:
# ============================================================
# CELL 2 — Dataset
# ============================================================
SYSTEM_PROMPT = """You are an expert SVG coder. Generate valid, complete SVG code matching the user's description.
STRICT CONSTRAINTS:
1. Return ONLY raw SVG code. No markdown, no explanations.
2. Canvas must be exactly 256x256: width='256' height='256' viewBox='0 0 256 256'.
3. Use only allowed elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, desc, style, pattern, marker, filter.
4. Maximum 8000 characters. Maximum 256 path elements.
5. No scripts, event handlers, animation, foreignObject, or external references.
6. Ensure all tags are properly closed and XML is well-formed."""

def format_example(example):
    return {"text": (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nGenerate SVG for: {example['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['svg']}<|im_end|>"
    )}

print("Loading clean dataset...")
dataset = load_dataset("csv", data_files=DATASET_PATH, split="train")
dataset = dataset.filter(lambda x: x["prompt"] is not None and x["svg"] is not None)
dataset = dataset.map(format_example, remove_columns=dataset.column_names, num_proc=4)

# Train / val split
split        = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"✅ Train: {len(train_dataset)} | Val: {len(eval_dataset)}")
print(f"Sample:\n{train_dataset[0]['text']}...")

Loading clean dataset...
✅ Train: 31350 | Val: 1650
Sample:
<|im_start|>system
You are an expert SVG coder. Generate valid, complete SVG code matching the user's description.
STRICT CONSTRAINTS:
1. Return ONLY raw SVG code. No markdown, no explanations.
2. Canvas must be exactly 256x256: width='256' height='256' viewBox='0 0 256 256'.
3. Use only allowed elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, desc, style, pattern, marker, filter.
4. Maximum 8000 characters. Maximum 256 path elements.
5. No scripts, event handlers, animation, foreignObject, or external references.
6. Ensure all tags are properly closed and XML is well-formed.<|im_end|>
<|im_start|>user
Generate SVG for: Generate svg code for an image that looks like: a gift box with a bow. Don't use markdown just give svg code<|im_end|>
<|im_start|>assistant
<svg width="200" height="250" viewBox="0 0 200 250" xml

In [4]:
# ============================================================
# CELL 3 — Load Model
# ============================================================
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name          = MODEL_NAME,
    max_seq_length      = MAX_SEQ_LEN,
    load_in_4bit        = False,
    dtype               = torch.bfloat16,
    fast_inference      = False,
    attn_implementation = "sdpa",
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,           # up from 16
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "embed_tokens", "lm_head"],
    lora_alpha     = 64,           # 2x rank
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = False,
    random_state   = 3407,
)

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

allocated = torch.cuda.memory_allocated()/1024**3
print(f"✅ Model loaded | VRAM used: {allocated:.2f} GB | Free: {24-allocated:.2f} GB")
print(f"Device: {model.device} | Dtype: {model.dtype}")

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.17 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
✅ Model loaded | VRAM used: 6.60 GB | Free: 17.40 GB
Device: cuda:0 | Dtype: torch.bfloat16


In [5]:
# ============================================================
# CELL 4 — Train
# ============================================================
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    packing            = True,
    dataset_num_proc   = 4,
    args = SFTConfig(
        # Batch & accumulation
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = 8,           # effective batch = 16

        # Learning rate & schedule
        learning_rate               = 1e-4,        # lower than v1
        warmup_steps                = 100,          # more warmup
        num_train_epochs            = 2,            # 2 epochs on clean data

        # Precision & optimizer
        bf16                        = True,
        optim                       = "adamw_8bit",
        seed                        = 3407,

        # Output
        output_dir                  = OUTPUT_DIR,

        # WSL2 dataloader settings
        dataloader_pin_memory       = False,
        dataloader_num_workers      = 0,
        gradient_checkpointing      = False,

        # Logging
        logging_steps               = 5,

        # Evaluation
        eval_strategy               = "steps",
        eval_steps                  = 200,

        # Checkpointing — save best model by val loss
        save_strategy               = "steps",
        save_steps                  = 200,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,

        report_to                   = "none",
    ),
)

print("🚀 Training v2...")
trainer_stats = trainer.train()
print(f"✅ Done in {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"Speed: {trainer_stats.metrics['train_samples_per_second']:.2f} samples/s")

Unsloth: Tokenizing ["text"] (num_proc=9):   0%|          | 0/31350 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=9):   0%|          | 0/1650 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Training v2...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,350 | Num Epochs = 2 | Total steps = 3,920
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 375,959,552 of 3,461,898,240 (10.86% trained)


Step,Training Loss,Validation Loss
200,0.379458,0.391996
400,0.340320,0.358462
600,0.329505,0.339958


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 5 — Save & Plot
# ============================================================
# Save LoRA adapter
adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✅ LoRA adapter saved to {adapter_path}")

# Save merged model
merged_path = os.path.join(OUTPUT_DIR, "full_model")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"✅ Merged model saved to {merged_path}")

# Verify both models exist
print(f"\nModel check:")
print(f"  v1 adapter: {os.path.exists('./qwen-svg-lora/lora_adapter')}")
print(f"  v2 adapter: {os.path.exists(adapter_path)}")

# Loss plot — train and val
logs        = trainer.state.log_history
train_steps = [l["step"] for l in logs if "loss" in l and "eval_loss" not in l]
train_loss  = [l["loss"] for l in logs if "loss" in l and "eval_loss" not in l]
eval_steps  = [l["step"] for l in logs if "eval_loss" in l]
eval_loss   = [l["eval_loss"] for l in logs if "eval_loss" in l]

plt.figure(figsize=(12, 5))
plt.plot(train_steps, train_loss, color="#3498db", linewidth=2, label="Train Loss")
if eval_steps:
    plt.plot(eval_steps, eval_loss, color="#e74c3c", linewidth=2,
             label="Val Loss", linestyle="--", marker="o")
plt.title("Training Loss — v2 (clean data, 2 epochs, r=32)")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, "loss_chart.png"))
plt.show()
print(f"✅ Loss chart saved to {OUTPUT_DIR}/loss_chart.png")